## Content-Based Recommender | Model 3 | Sentence Transformers with different multipliers for repeating metadata

### This code constructs a content-based recommender system using sentence embeddings and evaluates the effect of metadata weighting by repeating non-plot features in the text representation.

###### Inspired by: Simulating Ibtesam
###### Link: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system/notebook

In [1]:
# Imports
import pandas as pd
import numpy as np
import random
from collections import Counter
from ast import literal_eval
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install sentence-transformers

In [3]:
# Read csv
df1=pd.read_csv('../tmdb/tmdb_5000_credits.csv')
df2=pd.read_csv('../tmdb/tmdb_5000_movies.csv')

In [4]:
# Join two datasets on id column
df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [5]:
# Parse the stringified features into their corresponding python objects
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

#### Functions that will help extract required info from each feature

In [6]:
# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [7]:
# Test directors function
df2['director'] = df2['crew'].apply(get_director)
df2[['title', 'director']].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [8]:
# Returns the list top 3 elements or entire list; whichever is more.
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []

In [9]:
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [10]:
# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[culture clash, future, space war]","[Action, Adventure, Fantasy]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[ocean, drug abuse, exotic island]","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[spy, based on novel, secret agent]","[Action, Adventure, Crime]"


#### Convert names and keywords instances into lowercase and strip spaces between them so vectorizer doesn't get confused by multiple people with the same first or last name.

In [11]:
# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [12]:
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

## Get Model

In [13]:
# Load pre-trained neural model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1086.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Prepare for Evaluation

#### Test users and movies they enjoy

In [14]:
# dictionary of 5 users with difference preferences
# Each movie is considered a movie the user enjoyed

user_history = {
    # User 1: Classic gangster & crime dramas
    "user_1": [
        "The Godfather", "GoodFellas", "Scarface", "Pulp Fiction", "The Departed",
        "The Godfather: Part II", "Casino", "Donnie Brasco", "Once Upon a Time in America",
        "The Untouchables", "Road to Perdition", "Public Enemies", "Gangs of New York",
        "A History of Violence", "Eastern Promises", "The Town", "The Conformist",
        "Find Me Guilty", "Black Mass", "The Hills Have Eyes"
    ],

    # User 2: Blockbuster action & superhero movies
    "user_2": [
        "Avatar", "Titanic", "Avengers: Age of Ultron", "Guardians of the Galaxy", "Iron Man",
        "Thor", "Captain America: The First Avenger", "The Avengers", "Ant-Man",
        "The Incredible Hulk", "Captain America: Civil War", "Iron Man 2", "Iron Man 3",
        "Thor: The Dark World", "Batman Begins", "The Dark Knight", "The Dark Knight Rises",
        "Batman & Robin", "Batman Returns", "Batman v Superman: Dawn of Justice"
    ],

    # User 3: Musical & biographical movies
    "user_3": [
        "Chicago", "Moulin Rouge!", "8MM", "Amnesiac", "Grease",
        "Les Misérables", "Inception", "The Pursuit of Happyness", "The Hit List",
        "Singin' in the Rain", "The Sound of Music", "West Side Story", "Mary Poppins",
        "The Wizard of Oz", "Frozen", "Aladdin", "Cinderella", "The Nutcracker",
        "Alice in Wonderland", "The Broadway Melody"
    ],

    # User 4: Fantasy & young adult series
    "user_4": [
        "The Lord of the Rings: The Fellowship of the Ring", "The Hobbit: An Unexpected Journey",
        "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",
        "Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",
        "Harry Potter and the Prisoner of Azkaban", "The Hunger Games: Catching Fire",
        "The Hunger Games: Mockingjay - Part 2", "The Twilight Saga: New Moon",
        "The Twilight Saga: Eclipse", "The Twilight Saga: Breaking Dawn - Part 2",
        "Percy Jackson: Sea of Monsters", "Percy Jackson & the Olympians: The Lightning Thief",
        "Harry Potter and the Order of the Phoenix", "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe",
        "The Hobbit: The Desolation of Smaug", "The Hobbit: The Battle of the Five Armies",
        "The Adventures of Huck Finn", "Hellboy II: The Golden Army"
    ],

    # User 5: Horror & thriller movies
    "user_5": [
        "The Shining", "1408", "8 Days", "The Conjuring", "Insidious",
        "Sinister", "Annabelle", "Paranormal Activity 2", "Halloween: Resurrection", "Psycho",
        "Jaws", "Saw: The Final Chapter", "Scream 3", "Pet Sematary", "White Noise 2: The Light",
        "It Follows", "The Possession", "The Exorcist", "Evil Dead", "Restoration"
    ]
}

## Split data into known and unknown

In [15]:
# Split movies into training and testing to evaluate model (5 training, 15 testing)
train_test_split = {}
split_ratio = 5

# Same results
random.seed(7)

# list and dict to store each unknown movie for each user
#users = {}

for user, movies in user_history.items():
    
    # List of unknown movies
    unknown_movies = []
    
    # add movies to either known or unknown list
    known = random.sample(movies, split_ratio)
    unknown = [m for m in movies if m not in known]
    
    # Add known and unknow movies to dictionary for corresponding user
    train_test_split[user] = {"known": known, "unknown": unknown}
    
    # append unknown movies to list
    #unknown_movies.append(unknown)
    #users[user] = unknown_movies

# Print 
for user, split in train_test_split.items():
    print(f"{user}:")
    print("Known:", split["known"])
    print("Unknown:", split["unknown"])
    print()

user_1:
Known: ['Road to Perdition', 'The Departed', 'Gangs of New York', 'GoodFellas', 'Scarface']
Unknown: ['The Godfather', 'Pulp Fiction', 'The Godfather: Part II', 'Casino', 'Donnie Brasco', 'Once Upon a Time in America', 'The Untouchables', 'Public Enemies', 'A History of Violence', 'Eastern Promises', 'The Town', 'The Conformist', 'Find Me Guilty', 'Black Mass', 'The Hills Have Eyes']

user_2:
Known: ['Batman & Robin', 'Guardians of the Galaxy', 'Iron Man 2', 'Titanic', 'Captain America: The First Avenger']
Unknown: ['Avatar', 'Avengers: Age of Ultron', 'Iron Man', 'Thor', 'The Avengers', 'Ant-Man', 'The Incredible Hulk', 'Captain America: Civil War', 'Iron Man 3', 'Thor: The Dark World', 'Batman Begins', 'The Dark Knight', 'The Dark Knight Rises', 'Batman Returns', 'Batman v Superman: Dawn of Justice']

user_3:
Known: ['Moulin Rouge!', '8MM', 'The Wizard of Oz', 'The Nutcracker', 'Alice in Wonderland']
Unknown: ['Chicago', 'Amnesiac', 'Grease', 'Les Misérables', 'Inception', 'T

## Evaluation

In [16]:

def evaluate(train_test_split, get_recommendations, k=10):
    
    # results dictionary
    results = {}
    
    # Loop through dictionary of users and movies
    for user, data in train_test_split.items():
        
        # Store known and unknown movies
        known_movies   = data["known"]
        unknown_movies = data["unknown"]

        # List to store recommendations for each users movie
        all_recommendations = []
        
        # for every known movie (5 each)
        for movie in known_movies:
            
            # Get recommendations for each movie
            recs = get_recommendations(movie)
            
            if isinstance(recs, str):
                continue
            
            # Add recomendations to list of all recs for user
            all_recommendations.extend(recs.tolist())

        # Count how often each movie was recommended
        movie_counts = Counter(all_recommendations)
        
        # Take top 10 most frequently recommended movies
        top_k = [movie for movie, _ in movie_counts.most_common(k)]

        # Coiunt how many of the top 10 are actually in unknown set
        hits      = sum(1 for movie in top_k if movie in unknown_movies)
        precision = hits / k
        recall    = hits / len(unknown_movies)
        
        # Store results for each user
        results[user] = {"precision": precision, "recall": recall}

    # Calculate averages acress all users
    avg_p = sum(v["precision"] for v in results.values()) / len(results)
    avg_r = sum(v["recall"]    for v in results.values()) / len(results)
    
    # Print averages
    print(f"  Avg Precision@{k}: {avg_p:.2f} | Avg Recall@{k}: {avg_r:.2f}")
    
    # Return dictionary of results
    return results

In [17]:
# Combines different text fields into one string for embedding
# Repeats metadata (cast, genre, etc.) to balance against long plot text
def unified_text_weighted(x, repeat=2):

    # Store fields
    plot     = x['overview']           if isinstance(x['overview'],  str)  else ''
    cast     = ' '.join(x['cast'])     if isinstance(x['cast'],      list) else ''
    genres   = ' '.join(x['genres'])   if isinstance(x['genres'],    list) else ''
    keywords = ' '.join(x['keywords']) if isinstance(x['keywords'],  list) else ''
    director = x['director']           if isinstance(x['director'],  str)  else ''

    # Combine metadata
    meta = f"{cast} {director} {genres} {keywords}"

    # Repeat metadata multiple times to increase its weight
    return f"{plot} {' '.join([meta] * repeat)}".strip()

In [18]:
# Returns df with weighted text
def build_unified_text(df, repeat):
    return df.apply(lambda x: unified_text_weighted(x, repeat), axis=1)


def build_recommender(df, cosine_sim):
    
    # Create array of titles of movies
    indices = pd.Series(df.index, index=df['title']).drop_duplicates()
    
    # Lower the letter of all the movies
    indices_lower = pd.Series(indices.values, index=indices.index.str.lower())

    # Inner function to recommend movies
    def get_recommendations(title):
        
        # Make title lowercase
        title = title.lower()

        # If movie isn't in dataset, return none
        if title not in indices_lower:
            return None

        # Get index of movie
        idx = indices_lower[title]

        # Get similarity scores between this movie and all others. Store top 10
        sim_scores = sorted(
            enumerate(cosine_sim[idx]),
            key=lambda x: x[1],
            reverse=True
        )[1:11]

        # Extract movie indeices and return their titles
        return df['title'].iloc[[i[0] for i in sim_scores]]

    # Return the recommendation function
    return get_recommendations

In [19]:
# Run the experiment
def run_repetition_experiment(df, train_test_split, repeat=2, k=10):
    
    # Print experiment title
    print(f"\n=== Repetition x{repeat} ===")

    # Copy dataframe to avoid changing original
    df_exp = df.copy()
    
    # Get univied text and store into df
    df_exp['unified_text'] = build_unified_text(df_exp, repeat)

    # Convert text into embeddings
    embeddings = model.encode(df_exp['unified_text'].tolist(), batch_size=64)
    
    # Compute simiarity between all movies
    cosine_sim = cosine_similarity(embeddings, embeddings)

    # Build recommendations
    recommender = build_recommender(df_exp, cosine_sim)

    # Evaluate using precision and recall
    return evaluate(train_test_split, recommender, k=k)

In [20]:
# Run experiment for different weights
for r in [1, 2, 4, 6]:
    user_results = run_repetition_experiment(df2, train_test_split, repeat=r)
    print(user_results)


=== Repetition x1 ===
  Avg Precision@10: 0.32 | Avg Recall@10: 0.21

=== Repetition x2 ===
  Avg Precision@10: 0.28 | Avg Recall@10: 0.19

=== Repetition x4 ===
  Avg Precision@10: 0.26 | Avg Recall@10: 0.17

=== Repetition x6 ===
  Avg Precision@10: 0.26 | Avg Recall@10: 0.17


In [21]:
user_results = run_repetition_experiment(df2, train_test_split, repeat=1)
print(user_results)


=== Repetition x1 ===
  Avg Precision@10: 0.32 | Avg Recall@10: 0.21
{'user_1': {'precision': 0.4, 'recall': 0.26666666666666666}, 'user_2': {'precision': 0.5, 'recall': 0.3333333333333333}, 'user_3': {'precision': 0.2, 'recall': 0.13333333333333333}, 'user_4': {'precision': 0.4, 'recall': 0.26666666666666666}, 'user_5': {'precision': 0.1, 'recall': 0.06666666666666667}}
